In [0]:
spark.conf.set(
  "fs.azure.account.key.ayushdamg7370.dfs.core.windows.net",
  "xiKI4QF0gMqaJ7OZPoVraXUoU9muR4hjI3yIhpv7bVqJ0GanpCmhG423ZypkxZifOZYPzn4/EPpi+AStj4q6cw=="  
)


In [0]:
df_chicago = spark.read.option("header", True).option("delimiter", "\t").csv(
  "abfss://silver@ayushdamg7370.dfs.core.windows.net/Chicago_All_Years_Combined.tsv"
)


In [0]:
display(df_chicago)


Inspection_ID,DBA_Name,AKA_Name,License_#,Facility_Type,Risk,Address,City,State,Zip,Inspection_Date,Inspection_Type,Results,Violations,Latitude,Longitude,Location
2609909,HAPPY MARKET,HAPPY MARKET,2912802,Grocery Store,Risk 2 (Medium),2334 S WENTWORTH AVE,CHICAGO,IL,60616,2025-01-02,Canvass,Pass w/ Conditions,"2. CITY OF CHICAGO FOOD SERVICE SANITATION CERTIFICATE - Comments: UPON ARRIVAL, OBSERVED NO CITY OF CHICAGO CERTIFIED FOOD MANAGER ON SITE WHILE OPEN AND OPERATING. INSTRUCTED A CITY OF CHICAGO CERTIFIED FOOD MANAGER MUST BE ON SITE AT ALL TIMES WHILE OPEN AND OPERATING. PRIORITY FOUNDATION VIOLATION 7-38-012, CITATION ISSUED | 10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: 6-301.14: OBSERVED NO HANDWASHING SIGNS AT EXPOSED HANDSINK IN THE BASEMENT OR IN EMPLOYEE WASHROOM; INSTRUCTED TO PROVIDE. | 38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: 6-501.112: OBSERVED DEAD ROACHES INSIDE THE MOP SINK BASIN IN THE BUTCHER AREA, STICKY TRAPS IN BASEMENT FLOORS, ON THE WALL NEAR THE HOOD IN THE BASEMENT PREP AREA. INSTRUCTED TO REMOVE DEAD ROACHES AND MAINTAIN. | 49. NON-FOOD/FOOD CONTACT SURFACES CLEAN - Comments: OBSERVED SHELF UNDER FOOD PREP TABLE BEHIND BUTCHER AREA WITH RUST. ALSO, OBSERVED DEBRIS ON THE SHELVES IN THE SHOPPING AISLES. INSTRUCTED TO REMOVE RUST AND DEBRIS AND MAKE SURFACE SMOOTH AND EASILY CLEANABLE.",41.84995400192252,-87.63209419559098,"(41.84995400192252, -87.63209419559098)"
2609927,SAT KAIVAL FOOD INC/SUBWAY,SAT KAIVAL FOOD INC/SUBWAY,2728400,Restaurant,Risk 1 (High),1916 S STATE ST,CHICAGO,IL,60616,2025-01-02,Canvass,Pass,"36. THERMOMETERS PROVIDED & ACCURATE - Comments: OBSERVED NO AMBIENT AIR THERMOMETER INSIDE THE COLD HOLDING UNITS ON SITE. INSTRUCTED TO PROVIDE AND MAINTAIN. | 53. TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED - Comments: OBSERVED THE TOILET SEAT AND TOILET FIXTURES IN THE WOMEN'S TOILET ROOM IN NEED OF CLEANING. DRIED BODILY FLUID OBSERVED ON THE MENTIONED FIXTURES. INSTRUCTED TO CLEAN AND MAINTAIN.",41.85605269621059,-87.62731125804903,"(41.85605269621059, -87.62731125804903)"
2610482,"DANCEN GRILL RESTAURANT, INC.",DANCEN,2341747,Restaurant,Risk 1 (High),5114 N LINCOLN AVE,CHICAGO,IL,60625,2025-01-14,Non-Inspection,No Entry,None reported,41.97452873707269,-87.69233925845519,"(41.97452873707269, -87.69233925845519)"
2610081,"MARYS TAQUERIA AND GROCERY STORE, INC.",MARYS FOOD AND GROCERY STORE,2326530,Restaurant,Risk 1 (High),1901 S CANALPORT AVE,CHICAGO,IL,60616,2025-01-06,Canvass,Pass,"10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: OBSERVED NO HANDWASHING SIGNAGE AT THE HANDWASHING SINK NEAR THE THREE COMPARTMENT SINK IN THE REAR. INSTRUCTED TO PROVIDE AND MAINTAIN. | 47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: OBSERVED THE LOWER SHELVES UNDER THE FOOD PREP TABLES IN THE REAR FOOD PREP AREA WITH SLIGHT DEBRIS. INSTRUCTED TO REMOVE DEBRIS AND MAINTAIN. | 51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICES - Comments: OBSERVED NO BACKFLOW PREVENTION DEVICE ON THE ICE MACHINE IN THE REAR FOOD PREP AREA. INSTRUCTED TO INSTALL AN ADEQUATE BACKFLOW PREVENTION DEVICE ON THE ICE MACHINE. | 53. TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED - Comments: OBSERVED NO LID FOR GARBAGE RECEPTACLE IN GENDER NEUTRAL TOILET ROOMS. INSTRUCTED TO PROVIDE FOR DISPOSAL OF FEMININE PRODUCTS. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: OBSERVED THE FOLLOWING IN NEED OF CLEANING: THE CREVICES OF THE WALLBASES THROUGHOUT REAR FOOD PREP AREA ESPECIALLY UNDER THE HOT HOLDING EQUIPMENT, THE FLOOR INSIDE THE WALK-IN COOLER, WALLS UNDER THE THREE COMPARTMENT SINK AND ON THE SIDE OF THE HOT HOLDING EQUIPMENT. INSTRUCTED TO CLEAN AND MAINTAIN.",41.85666310572283,-87.64178573004398,"(41.85666310572283, -87.64178573004398)"
2610172,KOTO,KOTO,2872350,Restaurant,Risk 1 (High),258 W 31ST ST,CHICAGO,IL,60616,2025-01-07,Canvass,Pass,"10. ADEQUATE H

In [0]:
df_chicago.coalesce(1).write.mode("overwrite").parquet(
    "abfss://silver@ayushdamg7370.dfs.core.windows.net/tmp_chicago_parquet"
)


In [0]:
files = dbutils.fs.ls("abfss://silver@ayushdamg7370.dfs.core.windows.net/tmp_chicago_parquet")
parquet_file = [f.path for f in files if f.name.endswith(".parquet")][0]
print(parquet_file)  # This is the full ABFS path to the actual parquet file


abfss://silver@ayushdamg7370.dfs.core.windows.net/tmp_chicago_parquet/part-00000-tid-2899538650905485935-c69f2181-8471-4efc-8147-d4a435447c31-12-1-c000.snappy.parquet


In [0]:
# Target clean filename in root of silver
target_path = "abfss://silver@ayushdamg7370.dfs.core.windows.net/Chicago_All_Years_Combined.parquet"

# Copy the single part file to final path
dbutils.fs.cp(parquet_file, target_path)


Out[12]: True

In [0]:
dbutils.fs.rm("abfss://silver@ayushdamg7370.dfs.core.windows.net/tmp_chicago_parquet", recurse=True)


Out[13]: True